# Make or Miss League?

In this project we will try to compare the predictive power of scoring statistics and non-scoring statistics on the outcome of an NBA basketball game.


We will create 3 models which we will use to predict the last 72 games of an NBA teams season, based on their first 10 games:

- Full-scope model (includes all statistics, including scoring)

- Scoring model (Only includes statistics about the team's offense)

- Non-scoring model (include all statistics except those relating to the team's offense)

Will non-scoring statistics be able to predict the success of an NBA team, or does winning in this league really come down to makes and misses?

SAMPLING NOTES:

What statistics we should use to predict any given game is kind of subjective. Using the season's cumulative statistics helps negate variance but could be unrepresentative of the actual current team if injuries or trades occurred.

To avoid this misrepresentation of teams, we can create a rolling average which uses the past 10 games statistics to predict the next game. This does introduce noise but may be more effective.

We can test which method works best.

SCORING MODEL NOTES:

Ideally, the model only reflects the scoring/offensive effectiveness of the team itself, and doesn't take in any variables which indicate defense or opponent's offensive success.

Essentially testing the predictive power of offensive effectiveness, independent of defense.

NON-SCORING MODEL NOTES:

Ideally, the model depicts everything about basketball except whether or not the team in question can put the ball in the basket. Completely exclude all variables indicating points or whether a basket was made (leads, ties, assists (assistPercentage can probably stay), shooting percentages, attempts etc.)

Opponent scoring stuff can probably stay

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

In [4]:
# Import NBA team statistics since Nov. 1st 1996

import kagglehub

# Download latest version
path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")

print("Path to dataset files:", path)

100%|██████████| 1.03G/1.03G [00:50<00:00, 21.8MB/s]

Extracting files...


Path to dataset files: C:\Users\ragha\.cache\kagglehub\datasets\eoinamoore\historical-nba-data-and-player-box-scores\versions\493


In [5]:
nba_team_stats_df = pd.read_csv(path + "/TeamStatisticsExtended.csv")

In [ ]:
nba_team_stats_df = nba_team_stats_df[nba_team_stats_df['gameType'] == 'Regular Season']
nba_team_stats_df.head(5)

<class 'pandas.core.frame.DataFrame'>
Index: 67430 entries, 164 to 79705
Columns: 104 entries, gameId to opponentOffensiveReboundPercentage
dtypes: float64(72), int64(24), object(8)
memory usage: 54.0+ MB


,gameId,seriesGameNumber,teamId,opponentTeamId,home,win,teamScore,opponentScore,seed,numMinutes,...,percentUnassisted2PointMade,percentAssisted3PointMade,percentUnassisted3PointMade,percentAssistedFieldGoalsMade,percentUnassistedFieldGoalsMade,freeThrowAttemptRate,opponentEffectiveFieldGoalPercentage,opponentFreeThrowAttemptRate,opponentTurnoverPercentage,opponentOffensiveReboundPercentage
count,6.743000e+04,0.0,6.743000e+04,6.743000e+04,67430.000000,67430.000000,67430.000000,67430.000000,36.000000,67430.000000,...,67430.000000,67430.000000,67430.000000,67430.000000,67430.000000,67430.000000,67430.000000,67430.000000,67430.000000,60512.000000
mean,2.232275e+07,NaN,1.610613e+09,1.610613e+09,0.500000,0.500000,102.237328,102.237328,2.277778,241.734391,...,0.470041,0.838227,0.154561,0.595606,0.404406,0.292289,0.504604,0.292289,0.151715,0.298012
std,2.916632e+06,NaN,8.613182e+00,8.613182e+00,0.500004,0.500004,14.052975,14.052975,1.111270,7.363335,...,0.119574,0.180610,0.166380,0.105095,0.105096,0.106777,0.068813,0.106777,0.040753,0.075621
min,2.000059e+07,NaN,1.610613e+09,1.610613e+09,0.000000,0.000000,0.000000,0.000000,1.000000,240.000000,...,0.038000,0.000000,0.000000,0.107000,0.031000,0.000000,0.234000,0.000000,0.011000,0.022000
25%,2.070058e+07,NaN,1.610613e+09,1.610613e+09,0.000000,0.000000,92.000000,92.000000,1.000000,240.000000,...,0.387000,0.750000,0.000000,0.526000,0.333000,0.217000,0.457000,0.217000,0.124000,0.245000
50%,2.140064e+07,NaN,1.610613e+09,1.610613e+09,0.500000,0.500000,102.000000,102.000000,2.000000,240.000000,...,0.469000,0.875000,0.125000,0.600000,0.400000,0.280000,0.500000,0.280000,0.150000,0.297000
75%,2.220078e+07,NaN,1.610613e+09,1.610613e+09,1.000000,1.000000,111.000000,111.000000,3.000000,240.000000,...,0.552000,1.000000,0.250000,0.667000,0.474000,0.355000,0.550000,0.355000,0.178000,0.348000
max,2.990119e+07,NaN,1.610613e+09,1.610613e+09,1.000000,1.000000,176.000000,176.000000,4.000000,340.000000,...,0.957000,1.000000,1.000000,0.969000,0.893000,1.016000,0.808000,1.016000,0.344000,0.658000


In [ ]:
nba_team_stats_df = nba_team_stats_df.drop(columns=['gameType', 'gameLabel', 'gameSubLabel', 'seriesGameNumber'])
nba_team_stats_df

,gameId,gameDateTimeEst,teamId,teamCity,teamName,opponentTeamId,opponentTeamCity,opponentTeamName,home,win,...,percentUnassisted2PointMade,percentAssisted3PointMade,percentUnassisted3PointMade,percentAssistedFieldGoalsMade,percentUnassistedFieldGoalsMade,freeThrowAttemptRate,opponentEffectiveFieldGoalPercentage,opponentFreeThrowAttemptRate,opponentTurnoverPercentage,opponentOffensiveReboundPercentage
164,22501193,2026-04-12 20:30:00,1610612741,Chicago,Bulls,1610612742,Dallas,Mavericks,0,0,...,0.524,1.000,0.000,0.577,0.423,0.179,0.630,0.270,0.162,0.365
165,22501193,2026-04-12 20:30:00,1610612742,Dallas,Mavericks,1610612741,Chicago,Bulls,1,1,...,0.533,0.955,0.045,0.673,0.327,0.270,0.538,0.179,0.116,0.288
166,22501194,2026-04-12 20:30:00,1610612745,Houston,Rockets,1610612763,Memphis,Grizzlies,1,1,...,0.647,0.786,0.214,0.479,0.521,0.286,0.479,0.105,0.126,0.179
167,22501194,2026-04-12 20:30:00,1610612763,Memphis,Grizzlies,1610612745,Houston,Rockets,0,0,...,0.423,0.615,0.385,0.590,0.410,0.105,0.524,0.286,0.107,0.452
168,22501195,2026-04-12 20:30:00,1610612740,New Orleans,Pelicans,1610612750,Minnesota,Timberwolves,0,0,...,0.625,1.000,0.000,0.444,0.556,0.355,0.562,0.483,0.102,0.308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79701,29600005,1996-11-01 19:30:00,1610612748,Miami,Heat,1610612737,Atlanta,Hawks,1,1,...,0.280,0.800,0.200,0.743,0.257,0.244,0.415,0.492,0.261,0.326
79702,29600007,1996-11-01 19:30:00,1610612754,Indiana,Pacers,1610612765,Detroit,Pistons,0,0,...,0.481,0.833,0.167,0.576,0.424,0.311,0.515,0.515,0.209,0.333
79703,29600007,1996-11-01 19:30:00,1610612765,Detroit,Pistons,1610612754,Indiana,Pacers,1,1,...,0.769,1.000,0.000,0.375,0.625,0.515,0.486,0.311,0.196,0.311
79704,29600001,1996-11-01 19:00:00,1610612741,Chicago,Bulls,1610612738,Boston,Celtics,0,1,...,0.317,0.000,1.000,0.667,0.333,0.432,0.500,0.274,0.189,0.327


In [ ]:
nba_team_stats_df.columns

Index(['gameId', 'gameDateTimeEst', 'teamId', 'teamCity', 'teamName',
       'opponentTeamId', 'opponentTeamCity', 'opponentTeamName', 'home', 'win',
       'teamScore', 'opponentScore', 'seed', 'numMinutes', 'assists', 'steals',
       'blocks', 'blocksAgainst', 'fieldGoalsMade', 'fieldGoalsAttempted',
       'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted',
       'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted',
       'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive',
       'reboundsTotal', 'reboundsTeam', 'foulsPersonal', 'personalFoulsDrawn',
       'turnovers', 'turnoversTeam', 'plusMinusPoints', 'q1Points', 'q2Points',
       'q3Points', 'q4Points', 'ot1Points', 'ot2Points', 'otAllPoints',
       'benchPoints', 'biggestLead', 'biggestScoringRun', 'leadChanges',
       'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
       'pointsSecondChance', 'timesTied', 'timeoutsRemaining', 'seasonWins',
       'seasonLosse

# THIS BELOW NEEDS TO BE COMBED THROUGH TO MAKE SURE ALL VARIABLES ARE CORRECT

FULL SCOPE MODEL NOTES:

We want pretty much every variable available, but I think we should eliminate variables which are proxies for winning or losing

NON-SCORING MODEL NOTES:

Ideally, the model depicts everything about basketball except whether or not the team in question can put the ball in the basket. Completely exclude all variables indicating points or whether a basket was made (leads, ties, assists (assistPercentage can probably stay), shooting percentages, attempts etc.)

Opponent scoring stuff can probably stay

SCORING MODEL NOTES:

Ideally, the model only reflects the scoring/offensive effectiveness of the team itself, and doesn't take in any variables which indicate defense or opponent's offensive success.

Essentially testing the predictive power of offensive effectiveness, independent of defense.

In [ ]:
# Metadata / identifiers which must be excluded from all models
metadata_cols = [
    'gameId', 'gameDateTimeEst', 'teamId', 'teamCity', 'teamName',
    'opponentTeamId', 'opponentTeamCity', 'opponentTeamName', 'seed',
    'numMinutes', 'timeoutsRemaining', 'seasonWins', 'seasonLosses'
]

# Target variable
target = 'win'

# Model 1: Full Scope
full_scope_cols = [
    # home court advantage
    'home',
    # Scoring
    'teamScore', 'opponentScore',
    'q1Points', 'q2Points', 'q3Points', 'q4Points',
    'ot1Points', 'ot2Points', 'otAllPoints',
    'benchPoints', 'plusMinusPoints',
    'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
    'pointsSecondChance', 'pointsOffTurnovers',
    'opponentPointsOffTurnovers', 'opponentPointsSecondChance',
    'opponentPointsFastBreak', 'opponentPointsInPaint',
    # Shooting
    'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
    'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
    'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage',
    'effectiveFieldGoalPercentage', 'trueShootingPercentage',
    'freeThrowAttemptRate', 'opponentEffectiveFieldGoalPercentage',
    'opponentFreeThrowAttemptRate',
    'percentFieldGoalAttempts2Point', 'percentFieldGoalAttempts3Point',
    'percentPoints2Point', 'percentPoints2PointMidRange',
    'percentPoints3Point', 'percentPointsFastBreak',
    'percentPointsFreeThrow', 'percentPointsOffTurnovers',
    'percentPointsInPaint',
    'percentAssisted2PointMade', 'percentUnassisted2PointMade',
    'percentAssisted3PointMade', 'percentUnassisted3PointMade',
    'percentAssistedFieldGoalsMade', 'percentUnassistedFieldGoalsMade',
    # Non-scoring
    'assists', 'steals', 'blocks', 'blocksAgainst',
    'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'reboundsTeam',
    'foulsPersonal', 'personalFoulsDrawn',
    'turnovers', 'turnoversTeam',
    'biggestLead', 'biggestScoringRun', 'leadChanges', 'timesTied',
    # Advanced
    'estimatedOffensiveRating', 'offensiveRating',
    'estimatedDefensiveRating', 'defensiveRating',
    'estimatedNetRating', 'netRating',
    'assistPercentage', 'assistToTurnoverRatio', 'assistRatio',
    'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage',
    'teamTurnoverPercentage', 'opponentTurnoverPercentage',
    'opponentOffensiveReboundPercentage',
    'estimatedPace', 'pace', 'pacePer40', 'possessions',
    'playerImpactEstimate'
]

# ── Model 2: Scoring Only ──────────────────────────────────────────────────────
scoring_cols = [
    'home',
    # Raw scoring
    'teamScore', #'opponentScore',
    'q1Points', 'q2Points', 'q3Points', 'q4Points',
    'ot1Points', 'ot2Points', 'otAllPoints',
    'benchPoints', #'plusMinusPoints',
    'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
    'pointsSecondChance', 'pointsOffTurnovers',
    #'opponentPointsOffTurnovers', 'opponentPointsSecondChance',
    #'opponentPointsFastBreak', 'opponentPointsInPaint',
    # Shooting
    'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
    'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
    'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage',
    'effectiveFieldGoalPercentage', 'trueShootingPercentage',
    'freeThrowAttemptRate', #'opponentEffectiveFieldGoalPercentage',
    #'opponentFreeThrowAttemptRate',
    'percentFieldGoalAttempts2Point', 'percentFieldGoalAttempts3Point',
    'percentPoints2Point', 'percentPoints2PointMidRange',
    'percentPoints3Point', 'percentPointsFastBreak',
    'percentPointsFreeThrow', 'percentPointsOffTurnovers',
    'percentPointsInPaint',
    'percentAssisted2PointMade', 'percentUnassisted2PointMade',
    'percentAssisted3PointMade', 'percentUnassisted3PointMade',
    'percentAssistedFieldGoalsMade', 'percentUnassistedFieldGoalsMade',
]



#THIS NEEDS MORE VARIABLES REGARDING OPPONENT SHOOTING STUFF
# ── Model 3: Non-Scoring Only
non_scoring_cols = [
    'home',
    # Hustle / defense
    'steals', 'blocks', 'blocksAgainst',
    'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'reboundsTeam',
    'foulsPersonal', 'personalFoulsDrawn',
    'turnovers', 'turnoversTeam',
    # Advanced non-scoring
    'assistPercentage', 'assistToTurnoverRatio', 'assistRatio',
    'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage',
    'teamTurnoverPercentage', 'opponentTurnoverPercentage',
    'opponentOffensiveReboundPercentage',
    'estimatedPace', 'pace', 'pacePer40', 'possessions',
]

**GENERAL NARRATIVE**

Current style of play is : offense is king -> 
* League is shifting towards optimizing pure offensive potential , Three point shooting, and high intensity offensive play
* 3 Models test independently for -> 
    * Full Scope model - Uses all columns to try and predicts win rate based off both offensive and non offensive metrics
    * Offensive (Scoring) Model -> Intends to isolate offense and win predictions (win rate)
    * Non Offensive -> Opposite columns from above offensive model to predict win rate

Our goal -> holistically analyze the data and model outputs across three in depth models, and compare results

Ask the question -> Is the NBA's craze over offensive efficacy truly warranted? (Is it the most optimizing metric)